In [5]:
"""
cyber_schema_core.py
--------------------
Neural Schema core for the Cybersecurity MAS.

Node types:
    attack   → DDoS, PortScan, Bot, SQLi, etc.
    port     → port_80, port_443, port_22, etc.
    action   → log_blocked_ip, quarantine_host, alert_admin, etc.
    verdict  → BLOCK, MONITOR, QUARANTINE, CLEAR, ALERT
    pattern  → escalation, recurring, coordinated, etc.

Edge relationships:
    attack  ──triggers──►       action
    attack  ──targets──►        port
    verdict ──follows──►        attack
    action  ──effective_for──►  attack
    attack  ──escalates_to──►   pattern
    port    ──associated_with──► attack
"""

import sqlite3
import json
import time
import numpy as np
import networkx as nx
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

DB_PATH = Path.cwd() / "cyber_schema.db"

# ── Known node definitions ─────────────────────────────────────────────────────
# Pre-seeded so schema starts with domain knowledge, not blank

ATTACK_NODES = [
    ("DDoS",                     "Distributed Denial of Service volumetric flood"),
    ("PortScan",                 "Attacker probing network for open ports"),
    ("Bot",                      "Compromised host communicating with C2"),
    ("FTP-Patator",              "Automated brute force against FTP service"),
    ("SSH-Patator",              "Automated brute force against SSH service"),
    ("DoS Hulk",                 "HTTP flood generating unique URLs"),
    ("DoS GoldenEye",            "HTTP DoS keeping connections alive"),
    ("DoS slowloris",            "Slow HTTP holding connections with partial requests"),
    ("DoS Slowhttptest",         "Slow HTTP exhausting server connection pool"),
    ("Infiltration",             "Attacker gained internal network access"),
    ("Heartbleed",               "OpenSSL memory leak via malformed TLS heartbeat"),
    ("Web Attack Brute Force",   "Automated login attempts against web app"),
    ("Web Attack XSS",           "Malicious scripts injected into web responses"),
    ("Web Attack Sql Injection", "Malicious SQL queries to manipulate database"),
    ("BENIGN",                   "Normal legitimate network traffic"),
]

ACTION_NODES = [
    ("log_blocked_ip",     "Block and log a suspicious IP address"),
    ("quarantine_host",    "Isolate a compromised host from network"),
    ("flag_suspicious_ip", "Flag IP as suspicious for monitoring"),
    ("alert_admin",        "Send alert to security administrator"),
    ("log_clear",          "Log traffic as verified benign"),
]

VERDICT_NODES = [
    ("BLOCK",      "Block the source IP immediately"),
    ("QUARANTINE", "Isolate the affected host"),
    ("MONITOR",    "Flag and watch for further activity"),
    ("ALERT",      "Alert admin without blocking"),
    ("CLEAR",      "Mark as benign, no action needed"),
]

PORT_NODES = [
    ("port_21",  "FTP — File Transfer Protocol"),
    ("port_22",  "SSH — Secure Shell remote access"),
    ("port_53",  "DNS — Domain Name System"),
    ("port_80",  "HTTP — Web traffic"),
    ("port_443", "HTTPS — Encrypted web traffic"),
    ("port_445", "SMB — Windows file sharing"),
    ("port_3389","RDP — Remote Desktop Protocol"),
    ("port_6667","IRC — Internet Relay Chat (C2 channel)"),
]


# ── Dataclass for incident context ────────────────────────────────────────────

@dataclass
class IncidentContext:
    """
    What the schema returns before an investigation starts.
    This gets injected into agent prompts.
    """
    attack_type:        str
    times_seen:         int           # how many times this attack was seen before
    last_verdict:       str           # what was decided last time
    last_verdict_approved: bool       # did human approve it?
    effective_actions:  list[str]     # actions that worked before
    associated_ports:   list[str]     # ports this attack usually targets
    pattern_flags:      list[str]     # escalation warnings e.g. "recurring", "coordinated"
    confidence_boost:   float         # schema confidence in recommendation (0-1)
    recommendation:     str           # schema's suggested verdict


class CyberSchemaCore:
    def __init__(self, db_path: Path = DB_PATH):
        self.db_path = db_path
        self.graph   = nx.DiGraph()
        self._init_db()
        self._seed_domain_knowledge()
        self._load_graph_from_db()

    # ── DB Init ────────────────────────────────────────────────────────────────

    def _init_db(self):
        with self._conn() as conn:
            conn.executescript("""
                CREATE TABLE IF NOT EXISTS nodes (
                    id          TEXT PRIMARY KEY,
                    label       TEXT NOT NULL,
                    node_type   TEXT NOT NULL,
                    description TEXT,
                    weight      REAL DEFAULT 1.0,
                    metadata    TEXT DEFAULT '{}',
                    created_at  REAL,
                    updated_at  REAL
                );

                CREATE TABLE IF NOT EXISTS edges (
                    id          INTEGER PRIMARY KEY AUTOINCREMENT,
                    source_id   TEXT NOT NULL,
                    target_id   TEXT NOT NULL,
                    relation    TEXT NOT NULL,
                    weight      REAL DEFAULT 1.0,
                    metadata    TEXT DEFAULT '{}',
                    created_at  REAL,
                    FOREIGN KEY (source_id) REFERENCES nodes(id),
                    FOREIGN KEY (target_id) REFERENCES nodes(id),
                    UNIQUE (source_id, target_id, relation)
                );

                CREATE TABLE IF NOT EXISTS incidents (
                    id              INTEGER PRIMARY KEY AUTOINCREMENT,
                    attack_type     TEXT NOT NULL,
                    port            TEXT,
                    confidence      REAL,
                    severity        TEXT,
                    verdict         TEXT,
                    actions_taken   TEXT,
                    human_approved  INTEGER DEFAULT 0,
                    outcome_notes   TEXT,
                    timestamp       REAL
                );

                CREATE INDEX IF NOT EXISTS idx_incidents_attack
                    ON incidents(attack_type);
                CREATE INDEX IF NOT EXISTS idx_incidents_port
                    ON incidents(port);
                CREATE INDEX IF NOT EXISTS idx_edges_source
                    ON edges(source_id);
            """)

    def _conn(self) -> sqlite3.Connection:
        return sqlite3.connect(self.db_path)

    # ── Domain Knowledge Seeding ───────────────────────────────────────────────

    def _seed_domain_knowledge(self):
        """
        Pre-populates schema with known cybersecurity domain knowledge.
        Only runs once — skips if nodes already exist.
        """
        with self._conn() as conn:
            existing = conn.execute(
                "SELECT COUNT(*) FROM nodes").fetchone()[0]
            if existing > 0:
                return  # already seeded

        print("[CyberSchema] Seeding domain knowledge...")
        now = time.time()

        # Add all known node types
        for name, desc in ATTACK_NODES:
            self._add_node_raw(f"attack_{name}", name, "attack", desc, now)

        for name, desc in ACTION_NODES:
            self._add_node_raw(f"action_{name}", name, "action", desc, now)

        for name, desc in VERDICT_NODES:
            self._add_node_raw(f"verdict_{name}", name, "verdict", desc, now)

        for name, desc in PORT_NODES:
            self._add_node_raw(name, name, "port", desc, now)

        # Seed known attack → port relationships
        known_attack_ports = {
            "attack_DDoS"                    : ["port_80", "port_443", "port_53"],
            "attack_DoS Hulk"                : ["port_80", "port_443"],
            "attack_DoS GoldenEye"           : ["port_80", "port_443"],
            "attack_DoS slowloris"           : ["port_80", "port_443"],
            "attack_DoS Slowhttptest"        : ["port_80", "port_443"],
            "attack_FTP-Patator"             : ["port_21"],
            "attack_SSH-Patator"             : ["port_22"],
            "attack_Bot"                     : ["port_80", "port_443", "port_6667"],
            "attack_Heartbleed"              : ["port_443"],
            "attack_Infiltration"            : ["port_445", "port_3389", "port_22"],
            "attack_Web Attack Brute Force"  : ["port_80", "port_443"],
            "attack_Web Attack XSS"          : ["port_80", "port_443"],
            "attack_Web Attack Sql Injection": ["port_80", "port_443"],
            "attack_PortScan"                : ["port_22", "port_80", "port_443"],
        }

        for attack_id, ports in known_attack_ports.items():
            for port_id in ports:
                self._add_edge_raw(
                    attack_id, port_id, "targets", 1.0, now)

        # Seed known attack → typical verdict relationships (weak prior)
        known_verdicts = {
            "attack_DDoS"                    : "verdict_BLOCK",
            "attack_DoS Hulk"                : "verdict_BLOCK",
            "attack_DoS GoldenEye"           : "verdict_BLOCK",
            "attack_DoS slowloris"           : "verdict_BLOCK",
            "attack_DoS Slowhttptest"        : "verdict_BLOCK",
            "attack_FTP-Patator"             : "verdict_BLOCK",
            "attack_SSH-Patator"             : "verdict_BLOCK",
            "attack_Bot"                     : "verdict_QUARANTINE",
            "attack_Heartbleed"              : "verdict_ALERT",
            "attack_Infiltration"            : "verdict_QUARANTINE",
            "attack_PortScan"                : "verdict_MONITOR",
            "attack_Web Attack Brute Force"  : "verdict_BLOCK",
            "attack_Web Attack XSS"          : "verdict_ALERT",
            "attack_Web Attack Sql Injection": "verdict_ALERT",
            "attack_BENIGN"                  : "verdict_CLEAR",
        }

        for attack_id, verdict_id in known_verdicts.items():
            self._add_edge_raw(
                attack_id, verdict_id, "typical_verdict", 1.0, now)

        print("[CyberSchema] Domain knowledge seeded successfully.")

    # ── Raw DB Writes (used only during seeding) ───────────────────────────────

    def _add_node_raw(self, node_id, label, node_type, desc, now):
        with self._conn() as conn:
            conn.execute("""
                INSERT OR IGNORE INTO nodes
                (id, label, node_type, description, weight, metadata, created_at, updated_at)
                VALUES (?, ?, ?, ?, 1.0, '{}', ?, ?)
            """, (node_id, label, node_type, desc, now, now))
        self.graph.add_node(
            node_id, label=label, node_type=node_type, weight=1.0)

    def _add_edge_raw(self, source, target, relation, weight, now):
        with self._conn() as conn:
            conn.execute("""
                INSERT OR IGNORE INTO edges
                (source_id, target_id, relation, weight, metadata, created_at)
                VALUES (?, ?, ?, ?, '{}', ?)
            """, (source, target, relation, weight, now))
        if self.graph.has_node(source) and self.graph.has_node(target):
            self.graph.add_edge(
                source, target, relation=relation, weight=weight)

    # ── Load graph from DB ─────────────────────────────────────────────────────

    def _load_graph_from_db(self):
        with self._conn() as conn:
            for row in conn.execute(
                    "SELECT id, label, node_type, weight FROM nodes"):
                self.graph.add_node(
                    row[0], label=row[1], node_type=row[2], weight=row[3])
            for row in conn.execute(
                    "SELECT source_id, target_id, relation, weight FROM edges"):
                self.graph.add_edge(
                    row[0], row[1], relation=row[2], weight=row[3])

    # ── Core: Observe an Incident ──────────────────────────────────────────────

    def observe_incident(self,
                         attack_type:    str,
                         port:           int,
                         confidence:     float,
                         severity:       str,
                         verdict:        str,
                         actions_taken:  list[str],
                         human_approved: bool,
                         outcome_notes:  str = ""):
        """
        Called after every incident completes.
        Updates node weights, edge weights, and incident history.
        This is how the schema learns.

        Args:
            attack_type   : e.g. "DDoS"
            port          : e.g. 80
            confidence    : ML confidence 0-1
            severity      : "CRITICAL", "HIGH", "MEDIUM", "LOW"
            verdict       : "BLOCK", "MONITOR", etc.
            actions_taken : ["log_blocked_ip", "alert_admin"]
            human_approved: whether human approved the verdict
            outcome_notes : any extra context
        """
        now        = time.time()
        attack_id  = f"attack_{attack_type}"
        port_id    = f"port_{port}"
        verdict_id = f"verdict_{verdict}"

        # ── 1. Bump attack node weight ─────────────────────
        self._bump_node_weight(attack_id)

        # ── 2. Bump port node if known ─────────────────────
        if self.graph.has_node(port_id):
            self._bump_node_weight(port_id)

        # ── 3. Strengthen attack → verdict edge ────────────
        if self.graph.has_node(verdict_id):
            self._strengthen_edge(attack_id, verdict_id, "observed_verdict")

        # ── 4. Strengthen attack → port edge ───────────────
        if self.graph.has_node(port_id):
            self._strengthen_edge(attack_id, port_id, "observed_on_port")

        # ── 5. Strengthen attack → action edges ────────────
        for action_name in actions_taken:
            action_id = f"action_{action_name}"
            if self.graph.has_node(action_id):
                self._strengthen_edge(attack_id, action_id, "action_taken")
                # If human approved, extra weight — means it worked
                if human_approved:
                    self._strengthen_edge(
                        attack_id, action_id, "approved_action", delta=1.5)

        # ── 6. Log to incidents table ──────────────────────
        with self._conn() as conn:
            conn.execute("""
                INSERT INTO incidents
                (attack_type, port, confidence, severity, verdict,
                 actions_taken, human_approved, outcome_notes, timestamp)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                attack_type, str(port), confidence, severity, verdict,
                json.dumps(actions_taken),
                1 if human_approved else 0,
                outcome_notes, now
            ))

        print(f"[CyberSchema] ✓ Incident observed: {attack_type} → {verdict}"
              f" (approved={human_approved})")

    # ── Core: Consult Schema Before Investigation ──────────────────────────────

    def consult(self, attack_type: str, port: int) -> IncidentContext:
        """
        Called BEFORE the investigator agent starts.
        Returns everything the schema knows about this attack type and port.

        This is what gets injected into agent prompts.
        """
        attack_id = f"attack_{attack_type}"
        port_id   = f"port_{port}"

        # ── How many times seen? ───────────────────────────
        times_seen = self._get_incident_count(attack_type)

        # ── What verdict was used most? ────────────────────
        last_verdict, last_approved = self._get_last_verdict(attack_type)

        # ── What actions were most effective? ─────────────
        effective_actions = self._get_effective_actions(attack_type)

        # ── What ports is this attack associated with? ─────
        associated_ports = self._get_associated_ports(attack_id)

        # ── Pattern detection ──────────────────────────────
        pattern_flags = self._detect_patterns(attack_type, port)

        # ── Confidence boost from schema ───────────────────
        # Higher weight on the verdict edge = more confident recommendation
        confidence_boost = self._get_verdict_confidence(attack_id, last_verdict)

        # ── Schema recommendation ──────────────────────────
        recommendation = self._recommend_verdict(attack_type, attack_id)

        return IncidentContext(
            attack_type          = attack_type,
            times_seen           = times_seen,
            last_verdict         = last_verdict,
            last_verdict_approved= last_approved,
            effective_actions    = effective_actions,
            associated_ports     = associated_ports,
            pattern_flags        = pattern_flags,
            confidence_boost     = confidence_boost,
            recommendation       = recommendation,
        )

    # ── Pattern Detection ──────────────────────────────────────────────────────

    def _detect_patterns(self, attack_type: str, port: int) -> list[str]:
        """
        Detects escalation and coordination patterns.
        Returns list of human-readable warning flags.
        """
        flags  = []
        with self._conn() as conn:
            rows = conn.execute("""
                SELECT timestamp, verdict, human_approved
                FROM incidents
                WHERE attack_type = ?
                ORDER BY timestamp DESC
                LIMIT 10
            """, (attack_type,)).fetchall()

        if not rows:
            return flags

        count = len(rows)

        # ── Recurring attack ───────────────────────────────
        if count >= 3:
            flags.append(f"RECURRING: {attack_type} seen {count} times — pattern established")

        # ── Frequency escalation ───────────────────────────
        if count >= 2:
            now      = time.time()
            week_ago = now - (7 * 24 * 3600)
            recent   = [r for r in rows if r[0] > week_ago]
            if len(recent) >= 3:
                flags.append(
                    f"ESCALATING: {len(recent)} incidents in last 7 days")

        # ── Blocking not working ───────────────────────────
        block_count = sum(1 for r in rows if r[1] == "BLOCK")
        if block_count >= 3 and count >= 3:
            flags.append(
                "PERSISTENT: IP blocking applied multiple times — "
                "attacker not deterred. Review firewall rules.")

        # ── Human kept approving same action ──────────────
        approved_count = sum(1 for r in rows if r[2] == 1)
        if approved_count >= 3:
            flags.append(
                f"CONFIRMED PATTERN: Human approved same response "
                f"{approved_count} times — high confidence in verdict")

        # ── Port specific ──────────────────────────────────
        with self._conn() as conn:
            port_hits = conn.execute("""
                SELECT COUNT(*) FROM incidents
                WHERE port = ?
            """, (str(port),)).fetchone()[0]

        if port_hits >= 3:
            flags.append(
                f"HOT PORT: port {port} flagged {port_hits} times — "
                f"consider port-level firewall rule")

        return flags

    # ── Graph Helpers ──────────────────────────────────────────────────────────

    def _bump_node_weight(self, node_id: str, delta: float = 1.0):
        if not self.graph.has_node(node_id):
            return
        with self._conn() as conn:
            conn.execute("""
                UPDATE nodes SET weight = weight + ?, updated_at = ?
                WHERE id = ?
            """, (delta, time.time(), node_id))
        self.graph.nodes[node_id]["weight"] = (
            self.graph.nodes[node_id].get("weight", 1.0) + delta)

    def _strengthen_edge(self, source: str, target: str,
                         relation: str, delta: float = 1.0):
        if not (self.graph.has_node(source) and self.graph.has_node(target)):
            return
        now = time.time()
        with self._conn() as conn:
            existing = conn.execute("""
                SELECT id, weight FROM edges
                WHERE source_id=? AND target_id=? AND relation=?
            """, (source, target, relation)).fetchone()

            if existing:
                conn.execute(
                    "UPDATE edges SET weight = weight + ? WHERE id = ?",
                    (delta, existing[0]))
                new_w = existing[1] + delta
            else:
                conn.execute("""
                    INSERT INTO edges
                    (source_id, target_id, relation, weight, metadata, created_at)
                    VALUES (?, ?, ?, ?, '{}', ?)
                """, (source, target, relation, delta, now))
                new_w = delta

        if self.graph.has_edge(source, target):
            self.graph[source][target]["weight"] = new_w
        else:
            self.graph.add_edge(source, target, relation=relation, weight=new_w)

    def _get_incident_count(self, attack_type: str) -> int:
        with self._conn() as conn:
            return conn.execute(
                "SELECT COUNT(*) FROM incidents WHERE attack_type = ?",
                (attack_type,)).fetchone()[0]

    def _get_last_verdict(self, attack_type: str) -> tuple[str, bool]:
        with self._conn() as conn:
            row = conn.execute("""
                SELECT verdict, human_approved FROM incidents
                WHERE attack_type = ?
                ORDER BY timestamp DESC LIMIT 1
            """, (attack_type,)).fetchone()
        if not row:
            return "UNKNOWN", False
        return row[0], bool(row[1])

    def _get_effective_actions(self, attack_type: str) -> list[str]:
        """Returns actions that were taken AND human-approved for this attack."""
        with self._conn() as conn:
            rows = conn.execute("""
                SELECT actions_taken FROM incidents
                WHERE attack_type = ? AND human_approved = 1
                ORDER BY timestamp DESC LIMIT 5
            """, (attack_type,)).fetchall()

        action_counts = {}
        for row in rows:
            try:
                actions = json.loads(row[0])
                for a in actions:
                    action_counts[a] = action_counts.get(a, 0) + 1
            except Exception:
                pass

        return sorted(action_counts, key=action_counts.get, reverse=True)

    def _get_associated_ports(self, attack_id: str) -> list[str]:
        results = []
        for neighbor in self.graph.successors(attack_id):
            if self.graph.nodes[neighbor].get("node_type") == "port":
                w = self.graph[attack_id][neighbor].get("weight", 1.0)
                results.append((neighbor, w))
        return [p for p, _ in sorted(results, key=lambda x: x[1], reverse=True)]

    def _get_verdict_confidence(self, attack_id: str,
                                verdict: str) -> float:
        """
        Returns how confident the schema is about the verdict.
        Based on edge weight relative to max possible.
        """
        if not verdict or verdict == "UNKNOWN":
            return 0.0
        verdict_id = f"verdict_{verdict}"
        if not self.graph.has_edge(attack_id, verdict_id):
            return 0.0
        weight = self.graph[attack_id][verdict_id].get("weight", 1.0)
        return min(weight / 10.0, 1.0)  # normalize to 0-1, cap at 1

    def _recommend_verdict(self, attack_type: str, attack_id: str) -> str:
        """
        Picks the verdict with the strongest observed edge.
        Falls back to typical_verdict if no observed data yet.
        """
        best_verdict = None
        best_weight  = 0.0

        for neighbor in self.graph.successors(attack_id):
            if self.graph.nodes[neighbor].get("node_type") != "verdict":
                continue
            edge_data = self.graph[attack_id][neighbor]
            rel    = edge_data.get("relation", "")
            weight = edge_data.get("weight", 0.0)
            # Observed verdicts outweigh typical verdicts
            boost  = 2.0 if rel == "observed_verdict" else 1.0
            if weight * boost > best_weight:
                best_weight  = weight * boost
                best_verdict = self.graph.nodes[neighbor].get("label", neighbor)

        return best_verdict or "MONITOR"

    # ── Public Utilities ───────────────────────────────────────────────────────

    def get_attack_history(self, attack_type: str, limit: int = 10) -> list[dict]:
        """Full incident history for an attack type."""
        with self._conn() as conn:
            rows = conn.execute("""
                SELECT attack_type, port, confidence, severity,
                       verdict, actions_taken, human_approved,
                       outcome_notes, timestamp
                FROM incidents WHERE attack_type = ?
                ORDER BY timestamp DESC LIMIT ?
            """, (attack_type, limit)).fetchall()
        results = []
        for r in rows:
            results.append({
                "attack_type"   : r[0], "port"          : r[1],
                "confidence"    : r[2], "severity"      : r[3],
                "verdict"       : r[4],
                "actions_taken" : json.loads(r[5] or "[]"),
                "human_approved": bool(r[6]),
                "outcome_notes" : r[7],
                "timestamp"     : r[8],
            })
        return results

    def summary(self) -> str:
        """Human-readable schema state."""
        with self._conn() as conn:
            total_incidents = conn.execute(
                "SELECT COUNT(*) FROM incidents").fetchone()[0]
            attack_counts   = conn.execute("""
                SELECT attack_type, COUNT(*) as c
                FROM incidents GROUP BY attack_type
                ORDER BY c DESC LIMIT 5
            """).fetchall()

        lines = [
            "═" * 50,
            "  CYBER NEURAL SCHEMA — STATE",
            "═" * 50,
            f"  Graph nodes     : {self.graph.number_of_nodes()}",
            f"  Graph edges     : {self.graph.number_of_edges()}",
            f"  Total incidents : {total_incidents}",
            "",
            "  Most seen attacks:",
        ]
        for attack, count in attack_counts:
            lines.append(f"    • {attack:<35} {count} incident(s)")

        lines.append("═" * 50)
        return "\n".join(lines)
#===============================================================================================================
"""
cyber_reasoning_engine.py
--------------------------
Reasoning layer for the Cyber Neural Schema.

Responsibilities:
  - Format schema context into agent-ready prompt strings
  - Detect escalation patterns
  - Compare current incident to past incidents
  - Generate pre-investigation briefing for Investigator Agent
  - Generate verdict guidance for Decision Agent
  - Generate trend summary for Synthesis Agent
"""



class CyberReasoningEngine:
    def __init__(self, schema: CyberSchemaCore):
        self.schema = schema

    # ── Pre-Investigation Briefing ─────────────────────────────────────────────
    # Injected at the TOP of the Investigator Agent's prompt

    def get_investigator_briefing(self,
                                  attack_type: str,
                                  port: int) -> str:
        """
        Returns a formatted string to inject into the
        Investigator Agent's system prompt.

        Gives the agent historical context BEFORE it starts
        calling investigation functions.
        """
        ctx = self.schema.consult(attack_type, port)

        if ctx.times_seen == 0:
            return (
                f"\n[NEURAL SCHEMA — PRE-INVESTIGATION BRIEFING]\n"
                f"This is the FIRST time {attack_type} has been observed.\n"
                f"No historical data available — investigate thoroughly.\n"
                f"Suggested starting point: lookup_threat_intel, then calculate_severity.\n"
            )

        # Build briefing
        lines = [
            "\n[NEURAL SCHEMA — PRE-INVESTIGATION BRIEFING]",
            f"Attack type     : {attack_type}",
            f"Times seen      : {ctx.times_seen} previous incident(s)",
            f"Last verdict    : {ctx.last_verdict}"
            + (" (HUMAN APPROVED ✓)" if ctx.last_verdict_approved else " (not approved)"),
        ]

        if ctx.effective_actions:
            lines.append(
                f"Effective actions: {', '.join(ctx.effective_actions)}")

        if ctx.associated_ports:
            port_labels = [p.replace("port_", "") for p in ctx.associated_ports[:4]]
            lines.append(f"Typical ports   : {', '.join(port_labels)}")

        if ctx.confidence_boost > 0:
            pct = int(ctx.confidence_boost * 100)
            lines.append(
                f"Schema confidence: {pct}% — based on past incident outcomes")

        # Pattern warnings — most important part
        if ctx.pattern_flags:
            lines.append("\n⚠  PATTERN WARNINGS:")
            for flag in ctx.pattern_flags:
                lines.append(f"   → {flag}")

        lines.append(
            f"\nSchema recommendation: {ctx.recommendation}")
        lines.append(
            "Use this context to focus your investigation. "
            "Do not repeat checks the schema has already confirmed.\n")

        return "\n".join(lines)

    # ── Decision Agent Guidance ────────────────────────────────────────────────
    # Injected into Decision Agent's prompt

    def get_decision_guidance(self,
                              attack_type: str,
                              port: int) -> str:
        """
        Returns verdict guidance for the Decision Agent
        based on what worked in past incidents.
        """
        ctx      = self.schema.consult(attack_type, port)
        history  = self.schema.get_attack_history(attack_type, limit=5)

        if ctx.times_seen == 0:
            return (
                f"\n[NEURAL SCHEMA — DECISION GUIDANCE]\n"
                f"No prior incidents of {attack_type}. "
                f"Use threat intel and severity to decide.\n"
            )

        lines = [
            "\n[NEURAL SCHEMA — DECISION GUIDANCE]",
            f"Past incidents    : {ctx.times_seen}",
            f"Recommended verdict: {ctx.recommendation}",
        ]

        if ctx.effective_actions:
            lines.append(
                f"Actions that worked: {', '.join(ctx.effective_actions)}")

        # Show last 3 outcomes
        if history:
            lines.append("\nRecent incident outcomes:")
            for h in history[:3]:
                approved = "✓ approved" if h["human_approved"] else "✗ rejected"
                lines.append(
                    f"  • {h['verdict']:<12} {approved}  "
                    f"(severity: {h['severity']}, port: {h['port']})")

        # Escalation flags go here too
        if ctx.pattern_flags:
            lines.append("\n⚠  ESCALATION FLAGS:")
            for flag in ctx.pattern_flags:
                lines.append(f"   → {flag}")
            lines.append(
                "\nConsider escalating response beyond standard verdict.")

        lines.append("")
        return "\n".join(lines)

    # ── Synthesis Trend Summary ────────────────────────────────────────────────
    # Injected into Synthesis Agent's prompt

    def get_synthesis_trend(self, attack_type: str, port: int) -> str:
        """
        Returns trend data for the Synthesis Agent so the
        report reflects patterns, not just single incidents.
        """
        ctx     = self.schema.consult(attack_type, port)
        history = self.schema.get_attack_history(attack_type, limit=10)

        if ctx.times_seen == 0:
            return (
                f"\n[NEURAL SCHEMA — TREND SUMMARY]\n"
                f"First occurrence of {attack_type}. No trend data yet.\n"
            )

        # Frequency in last 7 days
        week_ago = time.time() - (7 * 24 * 3600)
        recent   = [h for h in history if h["timestamp"] > week_ago]

        # Verdict distribution
        verdict_counts = {}
        for h in history:
            v = h["verdict"]
            verdict_counts[v] = verdict_counts.get(v, 0) + 1

        lines = [
            "\n[NEURAL SCHEMA — TREND SUMMARY]",
            f"Total incidents   : {ctx.times_seen}",
            f"Last 7 days       : {len(recent)} incident(s)",
            f"Verdict history   : " + ", ".join(
                f"{v}×{c}" for v, c in verdict_counts.items()),
        ]

        if ctx.pattern_flags:
            lines.append("\nActive pattern flags:")
            for flag in ctx.pattern_flags:
                lines.append(f"  → {flag}")

        lines.append(
            "\nInclude trend context in your report. "
            "This is not an isolated incident.\n")

        return "\n".join(lines)

    # ── Post-Incident Update ───────────────────────────────────────────────────

    def update_from_incident(self,
                             attack_type:    str,
                             port:           int,
                             confidence:     float,
                             severity:       str,
                             verdict:        str,
                             actions_taken:  list[str],
                             human_approved: bool,
                             outcome_notes:  str = ""):
        """
        Called after every pipeline completes.
        Updates the schema with what happened.
        Simple wrapper around schema.observe_incident().
        """
        self.schema.observe_incident(
            attack_type    = attack_type,
            port           = port,
            confidence     = confidence,
            severity       = severity,
            verdict        = verdict,
            actions_taken  = actions_taken,
            human_approved = human_approved,
            outcome_notes  = outcome_notes,
        )

    # ── Utility: Format context for a quick print ──────────────────────────────

    def explain_consultation(self, attack_type: str, port: int) -> str:
        """Print a full consultation result — useful for debugging."""
        ctx = self.schema.consult(attack_type, port)
        lines = [
            "─" * 50,
            f"  Schema Consultation: {attack_type} on port {port}",
            "─" * 50,
            f"  Times seen          : {ctx.times_seen}",
            f"  Last verdict        : {ctx.last_verdict}",
            f"  Last approved       : {ctx.last_verdict_approved}",
            f"  Effective actions   : {ctx.effective_actions}",
            f"  Associated ports    : {ctx.associated_ports}",
            f"  Schema confidence   : {ctx.confidence_boost:.2f}",
            f"  Recommendation      : {ctx.recommendation}",
        ]
        if ctx.pattern_flags:
            lines.append("  Pattern flags:")
            for f in ctx.pattern_flags:
                lines.append(f"    ⚠  {f}")
        lines.append("─" * 50)
        return "\n".join(lines)

#===============================================================================================================
"""
check_cyber_schema.py
---------------------
Tests the cyber neural schema end to end.
Simulates a realistic week of incidents and verifies
the schema learns, escalates, and gives correct guidance.
"""

import sys
import os


# Clean slate — remove old DB if exists
db_file = os.path.join(os.getcwd(), "cyber_schema.db")
if os.path.exists(db_file):
    os.remove(db_file)



PASS  = "\033[92m  PASS\033[0m"
FAIL  = "\033[91m  FAIL\033[0m"
HEAD  = "\033[96m\033[1m"
WARN  = "\033[93m"
RESET = "\033[0m"

results = []

def check(label: str, condition: bool, detail: str = ""):
    status = PASS if condition else FAIL
    print(f"{status}  {label}" + (f"  →  {detail}" if detail else ""))
    results.append((label, condition))

def section(title: str):
    print(f"\n{HEAD}{'─'*55}")
    print(f"  {title}")
    print(f"{'─'*55}{RESET}")


# ══════════════════════════════════════════════════════════
#  INIT
# ══════════════════════════════════════════════════════════
section("1. Schema Initialization")

schema = CyberSchemaCore()
engine = CyberReasoningEngine(schema)

check("Schema created",
      schema is not None)
check("Graph has attack nodes",
      any(d["node_type"] == "attack"
          for _, d in schema.graph.nodes(data=True)))
check("Graph has verdict nodes",
      any(d["node_type"] == "verdict"
          for _, d in schema.graph.nodes(data=True)))
check("Graph has port nodes",
      any(d["node_type"] == "port"
          for _, d in schema.graph.nodes(data=True)))
check("DDoS → port_80 edge seeded",
      schema.graph.has_edge("attack_DDoS", "port_80"))
check("DDoS → typical BLOCK seeded",
      schema.graph.has_edge("attack_DDoS", "verdict_BLOCK"))
check("Bot → typical QUARANTINE seeded",
      schema.graph.has_edge("attack_Bot", "verdict_QUARANTINE"))


# ══════════════════════════════════════════════════════════
#  FIRST INCIDENT — schema has no history yet
# ══════════════════════════════════════════════════════════
section("2. First Incident (no history)")

ctx_first = schema.consult("DDoS", 80)

check("First consult — times_seen is 0",
      ctx_first.times_seen == 0)
check("First consult — last_verdict is UNKNOWN",
      ctx_first.last_verdict == "UNKNOWN")
check("First consult — no pattern flags",
      len(ctx_first.pattern_flags) == 0)
check("First consult — still gives a recommendation",
      ctx_first.recommendation != "")

briefing = engine.get_investigator_briefing("DDoS", 80)
check("First briefing says first time",
      "FIRST" in briefing.upper())

print(f"\n  First briefing preview:\n{briefing[:200]}...")


# ══════════════════════════════════════════════════════════
#  SIMULATE INCIDENT 1
# ══════════════════════════════════════════════════════════
section("3. Observing Incident 1 — DDoS, BLOCK, approved")

engine.update_from_incident(
    attack_type    = "DDoS",
    port           = 80,
    confidence     = 0.94,
    severity       = "CRITICAL",
    verdict        = "BLOCK",
    actions_taken  = ["log_blocked_ip", "alert_admin"],
    human_approved = True,
    outcome_notes  = "Blocked source IP, attack subsided"
)

ctx_after1 = schema.consult("DDoS", 80)

check("After incident 1 — times_seen is 1",
      ctx_after1.times_seen == 1)
check("After incident 1 — last_verdict is BLOCK",
      ctx_after1.last_verdict == "BLOCK")
check("After incident 1 — last_verdict approved",
      ctx_after1.last_verdict_approved is True)
check("After incident 1 — effective actions include log_blocked_ip",
      "log_blocked_ip" in ctx_after1.effective_actions)
check("After incident 1 — schema recommends BLOCK",
      ctx_after1.recommendation == "BLOCK")


# ══════════════════════════════════════════════════════════
#  SIMULATE INCIDENTS 2 AND 3
# ══════════════════════════════════════════════════════════
section("4. Observing Incidents 2 & 3 — same pattern")

engine.update_from_incident(
    attack_type    = "DDoS",
    port           = 80,
    confidence     = 0.91,
    severity       = "CRITICAL",
    verdict        = "BLOCK",
    actions_taken  = ["log_blocked_ip", "alert_admin"],
    human_approved = True,
    outcome_notes  = "Second DDoS this week"
)

engine.update_from_incident(
    attack_type    = "DDoS",
    port           = 80,
    confidence     = 0.88,
    severity       = "HIGH",
    verdict        = "BLOCK",
    actions_taken  = ["log_blocked_ip", "alert_admin"],
    human_approved = True,
    outcome_notes  = "Third DDoS — IP blocking repeated"
)

ctx_after3 = schema.consult("DDoS", 80)

check("After 3 incidents — times_seen is 3",
      ctx_after3.times_seen == 3)
check("Pattern flags detected after 3 incidents",
      len(ctx_after3.pattern_flags) > 0)
check("RECURRING flag raised",
      any("RECURRING" in f for f in ctx_after3.pattern_flags))
check("Schema confidence increased",
      ctx_after3.confidence_boost > 0)

print(f"\n  Pattern flags after 3 incidents:")
for flag in ctx_after3.pattern_flags:
    print(f"    {WARN}⚠  {flag}{RESET}")


# ══════════════════════════════════════════════════════════
#  SIMULATE INCIDENTS 4 AND 5 — blocking not deterring
# ══════════════════════════════════════════════════════════
section("5. Incidents 4 & 5 — blocking isn't working")

engine.update_from_incident(
    attack_type    = "DDoS",
    port           = 80,
    confidence     = 0.96,
    severity       = "CRITICAL",
    verdict        = "BLOCK",
    actions_taken  = ["log_blocked_ip", "alert_admin"],
    human_approved = True,
    outcome_notes  = "Fourth DDoS — new source IP despite previous blocks"
)

engine.update_from_incident(
    attack_type    = "DDoS",
    port           = 80,
    confidence     = 0.93,
    severity       = "CRITICAL",
    verdict        = "BLOCK",
    actions_taken  = ["log_blocked_ip", "alert_admin"],
    human_approved = True,
    outcome_notes  = "Fifth DDoS — coordinated attack suspected"
)

ctx_after5 = schema.consult("DDoS", 80)

check("After 5 incidents — times_seen is 5",
      ctx_after5.times_seen == 5)
check("PERSISTENT flag raised (blocking not working)",
      any("PERSISTENT" in f for f in ctx_after5.pattern_flags))
check("ESCALATING flag raised",
      any("ESCALATING" in f or "RECURRING" in f
          for f in ctx_after5.pattern_flags))
check("HOT PORT flag raised for port 80",
      any("HOT PORT" in f for f in ctx_after5.pattern_flags))
check("CONFIRMED PATTERN flag raised",
      any("CONFIRMED" in f for f in ctx_after5.pattern_flags))

print(f"\n  All pattern flags after 5 incidents:")
for flag in ctx_after5.pattern_flags:
    print(f"    {WARN}⚠  {flag}{RESET}")


# ══════════════════════════════════════════════════════════
#  INVESTIGATOR BRIEFING — after history built
# ══════════════════════════════════════════════════════════
section("6. Investigator Briefing — with full history")

briefing = engine.get_investigator_briefing("DDoS", 80)

check("Briefing mentions times seen",
      "5" in briefing)
check("Briefing mentions last verdict",
      "BLOCK" in briefing)
check("Briefing mentions pattern warnings",
      "PATTERN" in briefing.upper() or "PERSISTENT" in briefing)
check("Briefing mentions recommendation",
      "recommendation" in briefing.lower())

print(f"\n  Full investigator briefing:")
print(briefing)


# ══════════════════════════════════════════════════════════
#  DECISION GUIDANCE
# ══════════════════════════════════════════════════════════
section("7. Decision Agent Guidance")

guidance = engine.get_decision_guidance("DDoS", 80)

check("Guidance mentions past incidents",
      "5" in guidance)
check("Guidance mentions recommended verdict",
      "BLOCK" in guidance)
check("Guidance includes effective actions",
      "log_blocked_ip" in guidance)
check("Guidance includes escalation flags",
      "ESCALAT" in guidance.upper() or "PERSISTENT" in guidance.upper())

print(f"\n  Decision guidance preview:")
print(guidance[:400])


# ══════════════════════════════════════════════════════════
#  SYNTHESIS TREND
# ══════════════════════════════════════════════════════════
section("8. Synthesis Agent Trend Summary")

trend = engine.get_synthesis_trend("DDoS", 80)

check("Trend includes total incidents",
      "5" in trend)
check("Trend includes verdict history",
      "BLOCK" in trend)
check("Trend includes pattern flags",
      len(trend) > 100)

print(f"\n  Synthesis trend:")
print(trend)


# ══════════════════════════════════════════════════════════
#  DIFFERENT ATTACK — Bot, fresh history
# ══════════════════════════════════════════════════════════
section("9. Different Attack Type — Bot (fresh)")

ctx_bot = schema.consult("Bot", 443)

check("Bot — starts with 0 incidents",
      ctx_bot.times_seen == 0)
check("Bot — no pattern flags yet",
      len(ctx_bot.pattern_flags) == 0)

engine.update_from_incident(
    attack_type    = "Bot",
    port           = 443,
    confidence     = 0.87,
    severity       = "HIGH",
    verdict        = "QUARANTINE",
    actions_taken  = ["quarantine_host", "alert_admin"],
    human_approved = True,
    outcome_notes  = "C2 communication detected"
)

ctx_bot_after = schema.consult("Bot", 443)
check("Bot — after 1 incident, times_seen is 1",
      ctx_bot_after.times_seen == 1)
check("Bot — recommends QUARANTINE after approval",
      ctx_bot_after.recommendation == "QUARANTINE")
check("Bot — quarantine_host in effective actions",
      "quarantine_host" in ctx_bot_after.effective_actions)


# ══════════════════════════════════════════════════════════
#  FULL CONSULTATION EXPLAIN
# ══════════════════════════════════════════════════════════
section("10. Full Schema State")

print(schema.summary())

print(engine.explain_consultation("DDoS", 80))


# ══════════════════════════════════════════════════════════
#  RESULTS
# ══════════════════════════════════════════════════════════
total  = len(results)
passed = sum(1 for _, ok in results if ok)
failed = total - passed

print(f"\n{'═'*55}")
print(f"  RESULTS: {passed}/{total} checks passed")
if failed > 0:
    print(f"\033[91m  {failed} check(s) FAILED:\033[0m")
    for label, ok in results:
        if not ok:
            print(f"    ✗  {label}")
else:
    print(f"\033[92m  All checks passed — Cyber Schema is working.\033[0m")
print(f"{'═'*55}\n")

sys.exit(0 if failed == 0 else 1)



───────────────────────────────────────────────────────
  1. Schema Initialization
───────────────────────────────────────────────────────
[CyberSchema] Seeding domain knowledge...
[CyberSchema] Domain knowledge seeded successfully.
  PASS  Schema created
  PASS  Graph has attack nodes
  PASS  Graph has verdict nodes
  PASS  Graph has port nodes
  PASS  DDoS → port_80 edge seeded
  PASS  DDoS → typical BLOCK seeded
  PASS  Bot → typical QUARANTINE seeded

───────────────────────────────────────────────────────
  2. First Incident (no history)
───────────────────────────────────────────────────────
  PASS  First consult — times_seen is 0
  PASS  First consult — last_verdict is UNKNOWN
  PASS  First consult — no pattern flags
  PASS  First consult — still gives a recommendation
  PASS  First briefing says first time

  First briefing preview:

[NEURAL SCHEMA — PRE-INVESTIGATION BRIEFING]
This is the FIRST time DDoS has been observed.
No historical data available — investigate thoroughly

SystemExit: 0

C:\Users\HANK\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
